In [22]:
import os
import mlflow
import xgboost as xgb
import pandas as pd
import mlflow

In [23]:
uri = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")
mlflow.set_tracking_uri(uri)
mlflow.set_experiment("xgboost_cancer")

<Experiment: artifact_location='file:///c:/Users/320258215/Desktop/IA-ML/repaso_sklearn/notebooks_train/mlruns/1', creation_time=1772548345941, experiment_id='1', last_update_time=1772548345941, lifecycle_stage='active', name='xgboost_cancer', tags={}>

In [24]:
from sklearn.model_selection import train_test_split

col_target = "target"
df = pd.read_csv("../data/wdbc_clean.csv")


In [ ]:


def preprocess_data(clean_df, params_spliter, target_col="target", irrelevant_cols=None):

    from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder # <--- Agregamos LabelEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.model_selection import train_test_split

    le = LabelEncoder()
    clean_df[target_col] = le.fit_transform(clean_df[target_col])

    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    mlflow.log_param("target_mapping", mapping) # Verás {'B': 0, 'M': 1} en MLflow

    X_completo = clean_df.drop(columns=(irrelevant_cols or []))
    
    train_df, test_df = train_test_split(
        X_completo, 
        test_size=params_spliter["test_size"], 
        random_state=params_spliter["random_state"], 
        stratify=X_completo[target_col] if params_spliter["stratify"] else None
    )

    # MLflow no permite registrar datasets con columnas de tipo object, por eso es importante hacer el Label Encoding antes de registrar los datasets.
    # De esta manera, el target ya estará en formato numérico y MLflow podrá registrarlo sin problemas.
    ds_test = mlflow.data.from_pandas(test_df, targets=target_col, name="Test_Set")       # mlflow.data.from_pandas() nos asegura que el dataset esté en un formato compatible con MLflow.
    ds_training = mlflow.data.from_pandas(train_df, targets=target_col, name="Train_Set") # mlflow.data.from_pandas() nos asegura que el dataset esté en un formato compatible con MLflow.
    mlflow.log_input(ds_training, context="training")
    mlflow.log_input(ds_test, context="testing")

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]
    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col]

    mlflow.log_params({
        "test_size": params_spliter["test_size"],
        "random_state": params_spliter["random_state"],
        "stratify": params_spliter["stratify"]
    })

    num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
        ]
    ) 

    X_train_scaled = preprocessor.fit_transform(X_train)
    X_test_scaled = preprocessor.transform(X_test)

    # Loggear el preprocessor en MLFlow para poder usarlo luego en la inferencia, y así asegurar que los nuevos datos se transformen de la misma manera que los datos de entrenamiento.
    # Logmodel lo que hace es guardar el objeto preprocessor completo, con toda su información, para que luego pueda ser cargado y usado para transformar nuevos datos de la misma manera que los datos de entrenamiento.
    # Log artifac es para guardar archivos NO EJECUTABLES, como por ejemplo un archivo de texto con información adicional sobre el preprocessor, o un archivo de configuración con los parámetros usados para crear el preprocessor.
    mlflow.sklearn.log_model(preprocessor, artifact_path="preprocessor", registered_model_name="Preprocessor_XGBoost_Clasificacion")
    
    return X_train_scaled, X_test_scaled, y_train, y_test, preprocessor


In [ ]:
def balancear_clases(X_train, y_train):
    
    from imblearn.over_sampling import SMOTE
    # Esto es para balancear las clases en el conjunto de entrenamiento,
    #  generando nuevas muestras sintéticas de la clase minoritaria.
    #  Esto puede ayudar a mejorar el rendimiento del modelo si hay un desequilibrio significativo entre las clases.
    X_train_original = X_train.copy()
    y_train_original = y_train.copy()
    X_train, y_train = SMOTE().fit_resample(X_train, y_train)

        
    # 1. Calculamos los valores primero
    counts_before = y_train_original.value_counts().to_dict()
    counts_after = y_train.value_counts().to_dict()

    # 2. Logueamos números sueltos como METRICS
    mlflow.log_metrics({
        "num_samples_generated": len(X_train) - len(y_train_original)
    })

    # 3. Logueamos los diccionarios como PARAMS (porque no son números únicos)
    mlflow.log_params({
        "class_dist_before": str(counts_before),
        "class_dist_after": str(counts_after)
    })

    # Aquí contamos cuantas clases hay en cada conjunto, que cantidad de cada clase hay, y el porcentaje de cada clase. Esto es importante para entender la distribución de las clases en los datos de entrenamiento y prueba, y para detectar posibles problemas de desequilibrio de clases.
    print("Train class distribution:")
    print(y_train.value_counts())
    
    return X_train, y_train

In [32]:
    
def build_model(X_train, y_train, params):

    from sklearn.metrics import accuracy_score
        
    XGBoost_model = xgb.XGBClassifier(**params)
    mlflow.log_params(XGBoost_model.get_params())


    # TRAIN
    XGBoost_model.fit(X_train, y_train)
    accuracy_train = accuracy_score(y_train, XGBoost_model.predict(X_train))
    print("Training accuracy:", accuracy_train)
    
    # TEST
    y_pred = XGBoost_model.predict(X_test)
    accuracy_test = accuracy_score(y_test, y_pred)
    print("Test accuracy:", accuracy_test)

    # siempre mejor loggear test y val juntos.
    mlflow.log_metrics({
        "train_accuracy": accuracy_train,
        "test_accuracy": accuracy_test
    })  
      
    return XGBoost_model

In [33]:
with mlflow.start_run():

    params = {
        "n_estimators": 150,  
        "max_depth": 8,        
        "learning_rate": 0.05,
        "random_state": 42
    }

    params_spliter = {
        "test_size": 0.2, 
        "random_state": 42, 
        "stratify": True
    }
    
    irrelevant_cols = None

    target_col = "target"

    X_train, X_test, y_train, y_test, preprocessor = preprocess_data(df, target_col=target_col, irrelevant_cols=irrelevant_cols, params_spliter=params_spliter)

    X_train_balanced, y_train_balanced = balancear_clases(X_train, y_train)

    build_model(X_train_balanced, y_train_balanced, params=params)
    
    

c:\Users\320258215\AppData\Local\anaconda4\envs\heart-ml\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/03 21:07:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 21:07:23 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_functi

Train class distribution:
target
1    285
0    285
Name: count, dtype: int64
Training accuracy: 1.0
Test accuracy: 0.9736842105263158
